# Fraud Shield - Feature Engineering

Formalizes the features explored in `01_eda.ipynb` into reusable transforms
from `src/features/engineering.py` and `src/features/imbalance.py`, then
writes the processed dataset to `data/processed/` (and, later, to
SageMaker Feature Store).

**Inputs:** `data/raw/fraudTrain.csv`, `data/raw/fraudTest.csv`
**Outputs:** `data/processed/train_features.parquet`, `data/processed/test_features.parquet`


In [ ]:
import sys
sys.path.append('..')  # so `src` is importable from the notebooks/ folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.engineering import (
    add_geo_distance,
    add_transaction_velocity,
    add_spending_profile,
)
from src.features.imbalance import compute_class_weights

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)


## 1. Load raw data


In [ ]:
train = pd.read_csv('../data/raw/fraudTrain.csv')
test = pd.read_csv('../data/raw/fraudTest.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head()


## 2. Add an event-time / transaction id column

Needed for the rolling-window velocity feature, and later required by
SageMaker Feature Store (`record_identifier_name`, `event_time_feature_name`).


In [ ]:
def prep_base(df):
    df = df.copy()
    df = df.rename(columns={'trans_date_trans_time': 'datetime'})
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['event_time'] = df['datetime'].astype('int64') // 10**9  # unix seconds, required by Feature Store
    if 'trans_num' in df.columns:
        df['transaction_id'] = df['trans_num']
    else:
        df['transaction_id'] = df.index.astype(str)
    return df

train = prep_base(train)
test = prep_base(test)


## 3. Geo-distance feature

Haversine distance between cardholder location (`lat`/`long`) and
merchant location (`merch_lat`/`merch_long`).


In [ ]:
train = add_geo_distance(train)
test = add_geo_distance(test)

train[['lat', 'long', 'merch_lat', 'merch_long', 'geo_distance_km']].head()


## 4. Transaction velocity

Rolling count of transactions per card number in a trailing 1-hour window.
This requires the data sorted by `cc_num` and `datetime`, which
`add_transaction_velocity` handles internally.

**Note:** this is O(n log n) per card and can take a few minutes on the
full ~1.3M-row training set.


In [ ]:
train = add_transaction_velocity(train, window='1h')
test = add_transaction_velocity(test, window='1h')

train[['cc_num', 'datetime', 'amt', 'txn_velocity']].head(10)


## 5. Spending profile (per-cardholder amount z-score)


In [ ]:
train = add_spending_profile(train)
test = add_spending_profile(test)

train[['cc_num', 'amt', 'cc_avg_amt', 'cc_std_amt', 'amt_zscore']].head(10)


## 6. Check for NaN/inf introduced by feature engineering

`amt_zscore` will be NaN when a cardholder has only one transaction
(`cc_std_amt` = 0 or undefined). `txn_velocity` will be NaN for a card's
very first transaction if the rolling window has no prior data.


In [ ]:
engineered_cols = ['geo_distance_km', 'txn_velocity', 'cc_avg_amt', 'cc_std_amt', 'amt_zscore']
print('Train NaN counts:')
print(train[engineered_cols].isnull().sum())
print()
print('Test NaN counts:')
print(test[engineered_cols].isnull().sum())


In [ ]:
# Impute: 0 is a reasonable neutral fill for a first-ever transaction
# (no prior velocity, no prior spending profile deviation)
for df in (train, test):
    df['txn_velocity'] = df['txn_velocity'].fillna(1)
    df['amt_zscore'] = df['amt_zscore'].fillna(0)
    df['cc_std_amt'] = df['cc_std_amt'].fillna(0)

print('Remaining NaNs:')
print(train[engineered_cols].isnull().sum().sum(), '(train)')
print(test[engineered_cols].isnull().sum().sum(), '(test)')


## 7. Encode categorical features

`category` (merchant category) and `gender` are low-cardinality and
one-hot encode cleanly. High-cardinality fields (`merchant`, `job`, `city`,
`state`) are left out of the baseline feature set for now -- worth revisiting
as target-encoded features if the baseline models need more signal.


In [ ]:
categorical_cols = ['category', 'gender']

train_encoded = pd.get_dummies(train, columns=categorical_cols, prefix=categorical_cols)
test_encoded = pd.get_dummies(test, columns=categorical_cols, prefix=categorical_cols)

# Align columns in case a category appears in one split but not the other
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

print(f'Train encoded shape: {train_encoded.shape}')
print(f'Test encoded shape:  {test_encoded.shape}')


## 8. Final feature set

Select the columns that will feed the models: engineered features plus
the encoded categoricals and core numeric fields.


In [ ]:
feature_cols = (
    ['amt', 'city_pop', 'geo_distance_km', 'txn_velocity', 'amt_zscore']
    + [c for c in train_encoded.columns if c.startswith('category_') or c.startswith('gender_')]
)
id_cols = ['transaction_id', 'event_time', 'cc_num']
target_col = 'is_fraud'

train_final = train_encoded[id_cols + feature_cols + [target_col]].copy()
test_final = test_encoded[id_cols + feature_cols + [target_col]].copy()

print(f'Final feature count: {len(feature_cols)}')
train_final.head()


## 9. Class imbalance preview

Compute class weights for cost-sensitive training (used directly by
`scale_pos_weight` in XGBoost, `class_weight` in sklearn/PyTorch).
SMOTE/undersampling (`src/features/imbalance.py`) will be applied inside the
training notebook, fit only on the training fold to avoid leakage.


In [ ]:
weights = compute_class_weights(train_final[target_col])
print('Class weights:', weights)

fraud_rate = train_final[target_col].mean() * 100
print(f'Fraud rate: {fraud_rate:.4f}%')


## 10. Save processed features


In [ ]:
train_final.to_parquet('../data/processed/train_features.parquet', index=False)
test_final.to_parquet('../data/processed/test_features.parquet', index=False)

print('Saved train_features.parquet and test_features.parquet to data/processed/')


## 11. (Later) Push to SageMaker Feature Store

Once AWS credentials and `SAGEMAKER_ROLE_ARN` are configured (see `.env`),
ingest the engineered features so both training and the real-time inference
endpoint read from the same online/offline store.


In [ ]:
# from src.features.feature_store import get_or_create_feature_group, ingest
#
# feature_group = get_or_create_feature_group(train_final)
# ingest(feature_group, train_final)


## 12. Next steps

- Confirm feature distributions look sane post-imputation (rerun EDA-style
  plots on `train_final` if anything looks off)
- Move to `03_baseline_models.ipynb`: apply SMOTE/undersampling on the
  training fold only, then train LogReg / Random Forest / XGBoost
- Revisit high-cardinality categoricals (`merchant`, `job`, `state`) as
  target-encoded features if baseline recall is weak
